# Rejoin Phase 0 — Colab pilot (schema 2)

Validates the Phase 0 pipeline and collects escrow-survival **and escrow-alignment**
events. Auto-selects **Qwen3-4B + 0.6B** on a 16 GB T4 and **Qwen3-8B + 0.6B** when
Colab provides ~22 GB or more.

What changed from the first pilot:

- The code domain now **prefills** the ```python fence and the HumanEval
  signature+docstring, so the draft and target can no longer diverge on output
  format. That artifact was 27% of all events in the first run.
- Every event records `L_bridge` / `bridge_k` — whether the escrow reattaches after
  a short bridge. Offset-0 survival alone cannot distinguish "semantic pivot" from
  "resumes three tokens later".
- Raw `suffix_ids` and `realized_ids` are stored, so re-analysis never needs the GPU.
- Survival measurement is now **free** (no branch pass). `--branch-verify` is a
  cross-check for the smoke run only.
- The paranoid cache check fires per **cycle**. In the previous version it fired per
  prompt, so `--n 5 --paranoid 10` ran zero checks.

The pilot is deliberately small. Do not treat its statistics as a publication result,
and treat every `L_bridge` number as an **oracle upper bound**, not an achievable
online policy.

In [ ]:
# Colab already provides a CUDA-compatible PyTorch build. Avoid replacing it.
# Gradio is unused here and its Hub requirement conflicts with Transformers 4.x.
%pip uninstall -q -y gradio gradio-client
%pip install -q "transformers>=4.51,<5" datasets accelerate bitsandbytes

In [ ]:
import os, sys, json, subprocess
from datetime import datetime, timezone
from pathlib import Path

import torch

assert torch.cuda.is_available(), (
    "No GPU is attached. Runtime > Change runtime type > GPU, then reconnect."
)
gpu = torch.cuda.get_device_properties(0)
gpu_gib = gpu.total_memory / 1024**3
default_target = "Qwen/Qwen3-8B" if gpu_gib >= 21.5 else "Qwen/Qwen3-4B"
TARGET_OVERRIDE = None  # Set only to intentionally override auto-selection.
TARGET = TARGET_OVERRIDE or default_target
DRAFT = "Qwen/Qwen3-0.6B"

os.environ["HF_HOME"] = "/content/hf-cache"
print(f"GPU: {gpu.name} ({gpu_gib:.1f} GiB)")
print(f"Pilot pair: {TARGET} target + {DRAFT} draft")
if TARGET.endswith("8B") and gpu_gib < 24:
    print("The 8B pair may be tight here. If it OOMs, set TARGET_OVERRIDE to Qwen/Qwen3-4B.")

In [ ]:
# Persist traces across Colab disconnects.
from google.colab import drive
drive.mount("/content/drive")
TRACE_ROOT = Path("/content/drive/MyDrive/RejoinPhase0/traces")
TRACE_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Traces will be saved under {TRACE_ROOT}")

## Get the study code

The four files are versioned in the repo now, so runs are reproducible and the trace
records a git SHA. If the repo is private, either use a token in the URL or fall back
to `files.upload()` in the next cell.

In [ ]:
REPO = "https://github.com/cadelew/Rejoin-Speculative-Decoding.git"
WORK = Path("/content/rejoin")

if not WORK.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(WORK)], check=True)
else:
    subprocess.run(["git", "-C", str(WORK), "pull", "--ff-only"], check=True)

STUDY = WORK / "studies" / "phase0"
os.chdir(STUDY)
print("cwd:", os.getcwd())
print("files:", sorted(p.name for p in STUDY.glob("*.py")))
print("git:", subprocess.run(["git", "-C", str(WORK), "rev-parse", "--short", "HEAD"],
                             capture_output=True, text=True).stdout.strip())

In [ ]:
# Fallback ONLY if the clone above failed (private repo, no token).
# Select trace_core.py, test_core.py, spec_trace.py, analyze_traces.py together.
#
# from google.colab import files
# STUDY = Path("/content/phase0"); STUDY.mkdir(exist_ok=True); os.chdir(STUDY)
# uploaded = files.upload()
# required = {"trace_core.py", "test_core.py", "spec_trace.py", "analyze_traces.py"}
# assert not (required - set(uploaded)), f"missing: {sorted(required - set(uploaded))}"
pass

## 1. Validate the model-independent logic

Must print `ALL CORE TESTS PASSED`. This covers the off-by-one alignment, cache-crop bookkeeping, deferred pairing, bridge detection, and censoring.

In [ ]:
subprocess.run([sys.executable, "test_core.py"], check=True)

## 2. GPU plumbing self-test

Target and draft are both Qwen3-0.6B. Expect ~zero rejection events. The first model
download can take several minutes. Also read the printed output text — coherent text
matters as much as the rejection count.

In [ ]:
stamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
self_test_out = TRACE_ROOT / f"self-test-{stamp}.jsonl"
subprocess.run([
    sys.executable, "spec_trace.py",
    "--self-test", "--max-new", "64", "--out", str(self_test_out),
], check=True)
print(f"Self-test trace: {self_test_out}")

## 3. Smoke run — three independent cross-checks

This is the step the first pilot skipped, and it is the reason nothing in that run was
validated. It checks:

1. the KV cache against a cache-free recompute (`--paranoid 5`),
2. the free survival measurement against a dedicated branch pass (`--branch-verify`),
3. the cached draft against `model.generate` (`--check-draft-cache 3`).

All three must print `OK`.

In [ ]:
stamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
smoke_out = TRACE_ROOT / f"smoke-{TARGET.rsplit('/', 1)[-1]}-{stamp}.jsonl"
subprocess.run([
    sys.executable, "spec_trace.py",
    "--target", TARGET, "--draft", DRAFT,
    "--domain", "code", "--n", "5", "--gamma", "16", "--max-new", "256",
    "--paranoid", "5", "--branch-verify", "--check-draft-cache", "3",
    "--out", str(smoke_out),
], check=True)
print(f"Smoke trace: {smoke_out}")

## 4. Code-domain pilot

γ=16 rather than 32. With a 0.6B draft, γ=32 puts most events in the pathological
corner: the draft diverges at block index 0 and the resulting 31-token escrow was
generated with zero verified grounding. Sweep γ once the pipeline is trusted.

In [ ]:
DOMAIN = "code"
N_PROMPTS = 25
GAMMA = 16
MAX_NEW = 256

stamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
pilot_out = TRACE_ROOT / f"{DOMAIN}-{TARGET.rsplit('/', 1)[-1]}-g{GAMMA}-{stamp}.jsonl"
cmd = [
    sys.executable, "spec_trace.py",
    "--target", TARGET, "--draft", DRAFT,
    "--domain", DOMAIN, "--n", str(N_PROMPTS),
    "--gamma", str(GAMMA), "--max-new", str(MAX_NEW),
    "--paranoid", "25",
    "--out", str(pilot_out),
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)
print(f"Pilot trace: {pilot_out}")

## 5. Analyze

Read **`L_bridge` first** — it is the mechanism test. `L_survive` is the same test
restricted to the single worst attachment offset, which is what the first pilot
measured. Then read the economics block: break-even is
`g_min = cost_factor × baseline tokens-per-pass`, and only the batched cost model
gives recycle-if-better a plausible target.

In [ ]:
subprocess.run([sys.executable, "analyze_traces.py", str(pilot_out)], check=True)

In [ ]:
import pandas as pd
records = [json.loads(line) for line in pilot_out.read_text().splitlines() if line.strip()]
events = pd.DataFrame(r for r in records if r.get("type") == "event")
summaries = pd.DataFrame(r for r in records if r.get("type") == "summary")
print(f"Completed prompts: {len(summaries)} | paired events: {len(events)}")
if len(events):
    display(events[[
        "prompt_id", "step", "a", "m", "L_survive", "L_bridge", "bridge_k",
        "lcs_len", "L_fresh", "delta", "delta_bridge", "rejected", "correction",
    ]].head(30))
else:
    print("No paired rejection events. Increase N_PROMPTS or inspect the run output.")

### Eyeball the alignment cases

The events worth reading are the ones where the escrow reattaches after a bridge but
dies at offset 0 — `L_bridge > L_survive`. If that set is empty across a few hundred
events, the escrow premise is genuinely dead and no repair model can rescue it.

In [ ]:
if len(events):
    hits = events[events["L_bridge"] > events["L_survive"]]
    print(f"{len(hits)}/{len(events)} events reattach only after a bridge")
    for _, r in hits.head(8).iterrows():
        print("-" * 70)
        print(f"p{r.prompt_id} s{r.step}  m={r.m}  L_survive={r.L_survive} -> "
              f"L_bridge={r.L_bridge} at k={r.bridge_k}")
        print(f"  rejected  {r.rejected!r}  ->  correction {r.correction!r}")
        print(f"  escrow    {r.suffix_text!r}")
        print(f"  realized  {r.realized_text!r}")

## Stop here after the first pilot

Check that the three smoke-run cross-checks printed `OK`, that corrections and escrow
text look sensible, and that the censored fraction is small. Traces are already saved
to Drive. Do not launch hundreds of prompts until the schema, confidence intervals, and
resume behavior are fixed.

In [ ]:
# Optional local download in addition to the Google Drive copy.
# from google.colab import files; files.download(str(pilot_out))
print(pilot_out)